# 09 — RAG Problems in the Wild

Our RAG system works. But "works on 5 clean story files" and "works in production" are two very different claims. Every RAG system eventually runs into the same handful of failure modes. Let's hit each one on purpose, so you recognize them the first time they show up for real, instead of the fifth.

Six problems, in order: bad retrieval, lost in the middle, query-document mismatch, duplicate chunks, stale embeddings, and noisy documents. One bonus fix at the end: hybrid search.


In [ ]:
import os
import re
import json
import math
import faiss
import numpy as np
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"
embedder = SentenceTransformer("all-MiniLM-L6-v2")

DATA_DIR = os.path.join("..", "data")


def cosine_similarity(a, b) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x**2 for x in a))
    mag_b = math.sqrt(sum(x**2 for x in b))
    return dot / (mag_a * mag_b)


def embed_texts(texts: list[str]) -> np.ndarray:
    return embedder.encode(texts).astype(np.float32)


## First to break: wrong chunks come back

Say StoryVerse AI's catalog grows, and someone adds a story that happens to touch on chess. Embeddings understand meaning, not domain — "Dark Knight" the chess piece and "The Dark Knight" the movie both live in nearly the same semantic neighborhood as far as a general-purpose embedding model is concerned.


In [ ]:
ambiguous_docs = [
    {"content": "Batman is known as the Dark Knight, protector of Gotham City.", "metadata": {"source": "dark_knight.txt", "genre": "superhero"}},
    {"content": "In chess, the knight piece moves in an L-shape; a dark-squared knight controls dark squares.", "metadata": {"source": "chess_basics.txt", "genre": "chess"}},
]

embeddings = embed_texts([d["content"] for d in ambiguous_docs])
query = "Who is the Dark Knight?"
query_emb = embed_texts([query])[0]

for d, emb in zip(ambiguous_docs, embeddings):
    print(f"{cosine_similarity(query_emb, emb):.3f}  {d['metadata']['source']}")


Depending on the exact phrasing, the chess document can score close enough to genuinely compete with the Batman one — embeddings don't know your corpus has two totally different "Dark Knight"s in it. **Fix: metadata filtering.** Tag documents at indexing time and let a search be scoped to a category.


In [ ]:
import chromadb

chroma_client = chromadb.Client()
genre_collection = chroma_client.create_collection("genre_demo")
genre_collection.add(
    documents=[d["content"] for d in ambiguous_docs],
    embeddings=[e.tolist() for e in embeddings],
    metadatas=[d["metadata"] for d in ambiguous_docs],
    ids=[d["metadata"]["source"] for d in ambiguous_docs],
)

filtered = genre_collection.query(query_embeddings=[query_emb.tolist()], n_results=2, where={"genre": "superhero"})
print("Scoped to genre=superhero:", filtered["ids"])


When your corpus spans multiple domains, letting a caller (or the app) scope a query with a filter is often more reliable than hoping the embedding model resolves the ambiguity on its own.


## Fix that one, and here's the next thing that breaks

Metadata filtering handles the case where the *wrong* chunk gets retrieved. But what if the *right* chunk gets retrieved — and still doesn't get used properly, just because of where it happens to sit in the prompt? Same relevant chunk, three different positions, otherwise identical prompt. Does answer quality change with position alone?


In [ ]:
relevant_chunk = "Cooper's daughter is named Murph (Murphy). She solves the gravitational equation that saves Earth."
filler_chunks = [f"Filler fact #{i}: an unrelated detail about a different StoryVerse character." for i in range(6)]


def build_positioned_prompt(position: int) -> str:
    chunks = filler_chunks.copy()
    chunks.insert(position, relevant_chunk)
    context = "\n\n".join(chunks)
    return f"""Answer using ONLY this context. If not present, say you don't know.

Context:
{context}

Question: Who is Cooper's daughter, and what does she do?

Answer:"""


for position in [0, 3, 6]:
    prompt = build_positioned_prompt(position)
    response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.1)
    print(f"Relevant chunk at position {position}: {response.choices[0].message.content}\n")


Models tend to pay more attention to the very start and very end of a context, and less to the middle -- this is **lost in the middle**. If you see the position-3 (middle) answer come back vaguer or less complete than positions 0 and 6, that's the effect showing up directly.

**Fix: reranking.** Retrieve more candidates than you need, then reorder them so the strongest matches sit at the start and end, not buried in the middle.


In [ ]:
def rerank_chunks(scored_chunks: list[dict], top_n: int = 3) -> list[dict]:
    """Pick the top_n matches, then put the best first and the second-best last."""
    ranked = sorted(scored_chunks, key=lambda c: c["score"], reverse=True)[:top_n]
    if len(ranked) >= 3:
        return [ranked[0]] + ranked[2:] + [ranked[1]]
    return ranked


demo_scored = [{"content": f"chunk {i}", "score": s} for i, s in enumerate([0.4, 0.9, 0.6, 0.3, 0.8])]
print("Before rerank:", [(c["content"], c["score"]) for c in demo_scored])
print("After rerank: ", [(c["content"], c["score"]) for c in rerank_chunks(demo_scored)])


Production systems usually go further and use a **cross-encoder** (a small model trained specifically to score query-document relevance, like a Cohere Rerank or BGE reranker) instead of just reordering by the original score — cross-encoders are slower but noticeably more accurate at judging true relevance.


## Next: what if retrieval fails before it even starts?

The first two problems both assumed the right chunk was at least *findable*. Here's one where it might not be, because of how the question itself is phrased. A user asks casually: *"what's the twist in Interstellar?"* The actual matching sentence in our corpus reads formally: *"the tesseract was constructed by future humans to send gravitational data back through time."* Short and informal vs. long and formal — cosine similarity between the two can come back lower than you'd expect, purely from the style gap, not a meaning gap.


In [ ]:
casual_query = "what's the twist in Interstellar?"
formal_chunk = "The film's central revelation is that the tesseract was constructed by future humans to send gravitational data back through time."

casual_emb = embed_texts([casual_query])[0]
formal_emb = embed_texts([formal_chunk])[0]
print("Raw similarity:", round(cosine_similarity(casual_emb, formal_emb), 3))


**Fix 1 -- query expansion**: ask the model to rewrite the casual query into something closer to how the answer is actually phrased, before embedding it.


In [ ]:
def expand_query(question: str) -> str:
    prompt = f"""Rewrite this question to be more detailed and descriptive, as if searching a
formal plot summary. Return ONLY the rewritten question.

Original: {question}
Expanded:"""
    response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.2)
    return response.choices[0].message.content.strip()


expanded = expand_query(casual_query)
print("Expanded query:", expanded)
expanded_emb = embed_texts([expanded])[0]
print("Similarity after expansion:", round(cosine_similarity(expanded_emb, formal_emb), 3))


**Fix 2 -- HyDE (Hypothetical Document Embeddings)**: instead of rewriting the question, have the model *hallucinate* a plausible-sounding answer, and embed that. A fake-but-plausible answer is often stylistically much closer to the real chunk than the original question was.


In [ ]:
def hyde_embedding(question: str) -> np.ndarray:
    prompt = f"Write a short, plausible-sounding answer to this question, even if you're not sure it's correct:\n\n{question}"
    response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.3)
    hypothetical_answer = response.choices[0].message.content
    return embed_texts([hypothetical_answer])[0]


hyde_emb = hyde_embedding(casual_query)
print("Similarity using HyDE:", round(cosine_similarity(hyde_emb, formal_emb), 3))


Compare all three similarity scores above. Both fixes exist for the same reason: make the *query's* embedding look more like the *answer's* embedding stylistically, since that's what cosine similarity is actually comparing.


## A quieter one: duplicate chunks

Not every problem is about meaning at all — some are just plumbing bugs. A common, boring one: the same file gets loaded twice (a re-run of an ingestion script, a symlink, a copy-paste). Now your top-3 results are 3 copies of the same chunk, and you've spent your whole retrieval budget saying the same thing three times.


In [ ]:
interstellar_chunk = "Cooper falls into the black hole and reaches the tesseract, where he sends Murph the data that saves Earth."
duplicated_chunks = [
    {"content": interstellar_chunk, "embedding": embed_texts([interstellar_chunk])[0]},
    {"content": interstellar_chunk, "embedding": embed_texts([interstellar_chunk])[0]},
    {"content": "Harry stops Voldemort from obtaining the Sorcerer's Stone.", "embedding": embed_texts(["Harry stops Voldemort from obtaining the Sorcerer's Stone."])[0]},
]


def deduplicate_chunks(chunks: list[dict], similarity_threshold: float = 0.95) -> list[dict]:
    unique = []
    for chunk in chunks:
        is_duplicate = any(cosine_similarity(chunk["embedding"], u["embedding"]) > similarity_threshold for u in unique)
        if not is_duplicate:
            unique.append(chunk)
    return unique


print("Before dedup:", len(duplicated_chunks), "chunks")
print("After dedup: ", len(deduplicate_chunks(duplicated_chunks)), "chunks")


Even better than catching duplicates at retrieval time: catch them at *indexing* time with a content hash, so you never store the same chunk twice in the first place.


## A worse version of the same idea: stale embeddings

Duplicates are annoying but harmless. This next one is the same category of problem — the index doesn't match reality — except now it's actively wrong instead of just repetitive. Say you indexed 100 stories, then corrected the plot summary for 10 of them. If you don't update the index, the model keeps answering from the *old*, wrong text — silently, with no error to tip you off.


In [ ]:
stale_collection = chroma_client.create_collection("stale_demo")
stale_collection.add(
    documents=["Cooper's daughter is named Jessica."],  # deliberately wrong, simulating a stale entry
    embeddings=[embed_texts(["Cooper's daughter is named Jessica."])[0].tolist()],
    ids=["interstellar_chunk_1"],
)
print("Before fix:", stale_collection.get(ids=["interstellar_chunk_1"])["documents"])

# Fix: delete the stale entry by ID, then add the corrected version
stale_collection.delete(ids=["interstellar_chunk_1"])
stale_collection.add(
    documents=["Cooper's daughter is named Murph (Murphy)."],
    embeddings=[embed_texts(["Cooper's daughter is named Murph (Murphy)."])[0].tolist()],
    ids=["interstellar_chunk_1"],
)
print("After fix: ", stale_collection.get(ids=["interstellar_chunk_1"])["documents"])


**Fix strategies, cheapest to most thorough:** delete-and-re-add by a stable ID (what we just did), tag each entry with a version or timestamp so you can query "what's outdated," or just fully rebuild the index on a schedule if your corpus is small enough that this is cheap. In production, track document modification times and re-embed only what changed.


## Last one: what if the source text was never clean to begin with?

Every problem so far assumed our story text was well-written prose. Real-world sources rarely are. HTML tags, OCR garbage, repeated headers — all of it pollutes the embedding before chunking even gets a chance to help.


In [ ]:
noisy_document = """<html><body>INTERSTELLAR
INTERSTELLAR

C00per (sic) is a f0rmer NASA pil0t...
\x00\x00 PAGE 1 OF 22 \x00\x00
</body></html>"""


def clean_document(text: str) -> str:
    text = re.sub(r"<[^>]+>", "", text)       # strip HTML tags
    text = re.sub(r"\x00+", "", text)          # strip null bytes
    text = re.sub(r"\n{3,}", "\n\n", text)     # collapse excess blank lines
    text = re.sub(r" {2,}", " ", text)         # collapse repeated spaces
    return text.strip()


print("--- before ---")
print(repr(noisy_document))
print("\n--- after ---")
print(repr(clean_document(noisy_document)))


Always clean before chunking, and definitely before embedding. Garbage in the text means a garbage vector out -- there's no fixing that downstream.


## Six things broke, six things got fixed — here's the full list

| Problem | Root cause | Fix |
|---|---|---|
| Wrong chunks returned | Embeddings don't know your domain boundaries | Metadata filtering |
| Lost in the middle | Model attention favors the start/end of context | Reranking |
| Query-document mismatch | Query and answer differ in style/length | Query expansion or HyDE |
| Duplicate chunks | Indexing bug re-adds the same content | Deduplication (by hash or similarity) |
| Stale embeddings | No tracking of document updates | Delete + re-add by stable ID |
| Noisy documents | Poor source quality (HTML, OCR, artifacts) | A cleaning pass before chunking |


## One more, since you asked for six and semantic search has a blind spot

Semantic search alone misses exact matches that carry no "meaning" for an embedding model to latch onto -- IDs, codes, exact names. *"What happens in scene 42B?"* has no semantic content in "42B" at all.

**Hybrid search** combines a semantic score with a classic keyword score. Rather than reach for a new dependency, let's implement the keyword half ourselves -- it's the same **BM25** formula the popular `rank_bm25` package wraps, and seeing it once demystifies what "keyword search" actually means.


In [ ]:
def bm25_scores(query: str, documents: list[str], k1: float = 1.5, b: float = 0.75) -> list[float]:
    tokenized_docs = [doc.lower().split() for doc in documents]
    query_terms = query.lower().split()
    n_docs = len(documents)
    avg_len = sum(len(d) for d in tokenized_docs) / n_docs

    def idf(term):
        containing = sum(1 for d in tokenized_docs if term in d)
        return math.log(1 + (n_docs - containing + 0.5) / (containing + 0.5))

    scores = []
    for doc in tokenized_docs:
        score = 0.0
        doc_len = len(doc)
        for term in query_terms:
            freq = doc.count(term)
            if freq == 0:
                continue
            score += idf(term) * (freq * (k1 + 1)) / (freq + k1 * (1 - b + b * doc_len / avg_len))
        scores.append(score)
    return scores


documents = [
    "Cooper travels through the wormhole to reach scene 42B of the tesseract sequence.",
    "Harry Potter learns about the Sorcerer's Stone at Hogwarts.",
    "Scene 42B was the hardest sequence to film in the entire production.",
]

keyword_scores = bm25_scores("what happens in scene 42B", documents)
semantic_scores = [cosine_similarity(embed_texts(["what happens in scene 42B"])[0], embed_texts([d])[0]) for d in documents]

alpha = 0.5  # weight between semantic and keyword
for doc, k_score, s_score in zip(documents, keyword_scores, semantic_scores):
    final = alpha * s_score + (1 - alpha) * (k_score / (max(keyword_scores) or 1))
    print(f"final={final:.3f}  (semantic={s_score:.3f}, keyword={k_score:.2f})  {doc[:50]}...")


Notice the two documents literally containing "42B" score highly on keyword search even though their semantic content barely overlaps with anything -- exactly the case pure semantic search struggles with. Most production RAG systems run hybrid search rather than pure semantic, precisely to catch cases like this one.


## What we learned

Six real failure modes, and their fixes -- this is the difference between a demo and a system someone can actually rely on.

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Metadata filtering | Scoping a search to a category, using tags set at indexing time |
| Cross-encoder / reranker | A model that re-scores retrieved results for true relevance |
| Query expansion | Rewriting a query to better match how the answer is phrased |
| HyDE | Embedding a hypothetical (possibly wrong) answer instead of the raw question |
| BM25 | A classic keyword-matching relevance score |
| Hybrid search | Combining a semantic score and a keyword score into one ranking |

**Exercises:**

- Introduce a new noise pattern (repeated page numbers, say) and extend `clean_document` to handle it.
- Try `alpha` values of 0.2, 0.5, and 0.8 in the hybrid search cell -- when does keyword search start dominating the ranking?
- Deliberately create a stale-embedding scenario with one of your own StoryVerse stories, and verify the delete-and-re-add fix actually corrects it.
